## Testing different classification models

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [77]:
df = pd.read_csv("../data/processed/fraud_detection_dataset_preprocessed.csv")
df.head()

,Transaction_Amount,Account_Balance,Previous_Fraudulent_Activity,Card_Age,Transaction_Distance,Risk_Score,Fraud_Label,Transaction_Type_ATM Withdrawal,Transaction_Type_Bank Transfer,Transaction_Type_Online,...,Merchant_Category_Restaurants,Merchant_Category_Travel,Card_Type_Amex,Card_Type_Discover,Card_Type_Mastercard,Card_Type_Visa,Authentication_Method_Biometric,Authentication_Method_OTP,Authentication_Method_PIN,Authentication_Method_Password
0,39.79,93213.17,0,65,883.17,0.8494,0,False,False,False,...,False,True,True,False,False,False,True,False,False,False
1,1.19,75725.25,0,186,2203.36,0.0959,1,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,28.96,1588.96,0,226,1909.29,0.8400,1,False,False,True,...,True,False,False,False,False,True,True,False,False,False
3,254.32,76807.20,0,76,1311.86,0.7935,1,True,False,False,...,False,False,False,False,False,True,False,True,False,False
4,31.28,92354.66,1,140,966.98,0.3819,1,False,False,False,...,False,False,False,False,True,False,False,False,False,True


In [78]:
X = df.drop(columns=["Fraud_Label"])
y = df["Fraud_Label"]

In [79]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

In [80]:
from imblearn.over_sampling import SMOTE
sm = SMOTE()
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

In [81]:
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
X_train_res = pd.DataFrame(
    ss.fit_transform(X_train_res),
    columns=X.columns
)

X_test = pd.DataFrame(
    ss.transform(X_test),
    columns=X.columns
)

In [82]:
X_train_res.head()

,Transaction_Amount,Account_Balance,Previous_Fraudulent_Activity,Card_Age,Transaction_Distance,Risk_Score,Transaction_Type_ATM Withdrawal,Transaction_Type_Bank Transfer,Transaction_Type_Online,Transaction_Type_POS,...,Merchant_Category_Restaurants,Merchant_Category_Travel,Card_Type_Amex,Card_Type_Discover,Card_Type_Mastercard,Card_Type_Visa,Authentication_Method_Biometric,Authentication_Method_OTP,Authentication_Method_PIN,Authentication_Method_Password
0,-0.175374,1.686068,-0.282551,-0.400858,-1.008513,0.628548,1.529913,-0.652584,-0.654025,-0.655237,...,1.768325,-0.566737,-0.649410,1.527541,-0.659368,-0.655040,-0.656089,-0.652420,1.524946,-0.651733
1,-0.022652,0.386110,-0.282551,-0.823524,0.537427,-1.030949,-0.653632,-0.652584,-0.654025,1.526166,...,-0.565507,-0.566737,-0.649410,1.527541,-0.659368,-0.655040,1.524184,-0.652420,-0.655761,-0.651733
2,0.046415,0.297412,-0.282551,0.248236,1.458584,0.462006,1.529913,-0.652584,-0.654025,-0.655237,...,-0.565507,-0.566737,-0.649410,-0.654647,-0.659368,1.526624,-0.656089,1.532754,-0.655761,-0.651733
3,-1.034753,-0.171226,-0.282551,0.595425,-1.167975,1.405511,1.529913,-0.652584,-0.654025,-0.655237,...,-0.565507,1.764488,1.539859,-0.654647,-0.659368,-0.655040,-0.656089,-0.652420,-0.655761,1.534371
4,-0.927291,0.268318,3.539190,-0.883905,1.368329,1.238969,1.529913,-0.652584,-0.654025,-0.655237,...,-0.565507,-0.566737,-0.649410,-0.654647,-0.659368,1.526624,-0.656089,-0.652420,1.524946,-0.651733


In [83]:
y_train

8950     0
38421    0
19363    0
30157    1
14294    1
        ..
43723    0
32511    0
5192     1
12172    1
33003    0
Name: Fraud_Label, Length: 35000, dtype: int64

### Logistic Regression

In [84]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000, random_state=1)
lr.fit(X_train_res, y_train_res)
y_pred_lr = lr.predict(X_test)

In [85]:
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           0       0.80      0.91      0.85     10141
           1       0.74      0.51      0.61      4859

    accuracy                           0.78     15000
   macro avg       0.77      0.71      0.73     15000
weighted avg       0.78      0.78      0.77     15000



### Random Forest

In [86]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=300,
                            max_depth=15,
                            min_samples_split=5,
                            random_state=1)
rf.fit(X_train_res, y_train_res)
y_prob_rf = lr.predict_proba(X_test)[:,1]
y_pred_rf = (y_prob_rf > 0.3).astype(int)

In [87]:
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.80      0.61      0.69     10141
           1       0.45      0.68      0.54      4859

    accuracy                           0.63     15000
   macro avg       0.63      0.64      0.62     15000
weighted avg       0.69      0.63      0.64     15000



### Gradient Boosting

In [88]:
from sklearn.ensemble import GradientBoostingClassifier
gbc = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=1, random_state=1)
gbc.fit(X_train_res, y_train_res)
y_pred_gbc = gbc.predict(X_test)

In [89]:
print(classification_report(y_test, y_pred_gbc))

              precision    recall  f1-score   support

           0       0.80      1.00      0.89     10141
           1       1.00      0.47      0.64      4859

    accuracy                           0.83     15000
   macro avg       0.90      0.74      0.76     15000
weighted avg       0.86      0.83      0.81     15000



### Naive Bayes

In [90]:
from sklearn.naive_bayes import GaussianNB
nb = GaussianNB()
nb.fit(X_train_res, y_train_res)
y_pred_nb = nb.predict(X_test)

In [91]:
print(classification_report(y_test, y_pred_nb))

              precision    recall  f1-score   support

           0       0.78      0.82      0.80     10141
           1       0.58      0.51      0.54      4859

    accuracy                           0.72     15000
   macro avg       0.68      0.67      0.67     15000
weighted avg       0.71      0.72      0.72     15000



Recall, informuje jaki procent oszustw został wykryty przez model.
Precision mówi o jak często model miał rację, gdy przewidzi oszustwo.
W tym przypadku priorytezowany jest recall.  